# Carga de CSVs para tabelas

Este notebook automatiza a criação de tabelas a partir dos arquivos CSV da pasta `dados`, localizada ao lado do próprio notebook. Cada CSV é tratado como uma tabela independente.

O fluxo funciona assim:

* você informa o schema de destino
* escolhe quais tabelas deseja criar com base nos nomes dos arquivos
* define um prefixo opcional para os nomes finais das tabelas
* o notebook lê cada CSV, infere o schema e grava o resultado como tabela Delta

Exemplo de mapeamento:

* `customer_gold.csv` + prefixo `raw_` → `raw_customer_gold`
* `telco_gold.csv` + prefixo vazio → `telco_gold`


In [0]:
from pathlib import Path
import re

# Preencha apenas estas variáveis
CATALOGO = ""
SCHEMA = ""
PREFIXO = ""

base_dir = Path.cwd()
dados_dir = base_dir / "dados"
identificador_schema = f"{CATALOGO}.{SCHEMA}"

if not dados_dir.exists():
    raise FileNotFoundError(
        f"Pasta 'dados' não encontrada em: {dados_dir}"
    )

csv_files = sorted(dados_dir.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"Nenhum CSV encontrado em: {dados_dir}")


def normalizar_nome(nome):
    nome = re.sub(r"[^a-zA-Z0-9_]", "_", nome.strip().lower())
    nome = re.sub(r"_+", "_", nome).strip("_")
    return nome



resultados = []

for arquivo in csv_files:
    nome_tabela = normalizar_nome(f"{PREFIXO}{arquivo.stem}")
    tabela_destino = f"{identificador_schema}.{nome_tabela}"

    df = (
        spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(str(arquivo))
    )

    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", True)
        .saveAsTable(tabela_destino)
    )

    resultados.append(
        {
            "arquivo_csv": arquivo.name,
            "tabela_destino": tabela_destino,
            "linhas_gravadas": spark.table(tabela_destino).count(),
            "colunas": len(df.columns),
        }
    )

display(spark.createDataFrame(resultados))

arquivo_csv,colunas,linhas_gravadas,tabela_destino
credit_bureau_gold.csv,16,49440,serverless_stable_go25oh_catalog.workshop_schema.caroljfcredit_bureau_gold
credit_decisioning_features.csv,62,100000,serverless_stable_go25oh_catalog.workshop_schema.caroljfcredit_decisioning_features
customer_gold.csv,59,100000,serverless_stable_go25oh_catalog.workshop_schema.caroljfcustomer_gold
fund_trans_gold.csv,25,100000,serverless_stable_go25oh_catalog.workshop_schema.caroljffund_trans_gold
telco_gold.csv,10,53326,serverless_stable_go25oh_catalog.workshop_schema.caroljftelco_gold
